# 🌍 Phase 2 (Domain Adaptation): Mixed-Domain Fine-Tuning (PlantVillage + PlantDoc)

- **Vấn đề thực nghiệm (Domain Shift):** Mô hình `ConvNeXt-Tiny` (`EXP-03`) đạt **99.95% Macro-F1** trên phòng lab (`PlantVillage`) nhưng tụt xuống **30.93% Top-1 / 54.24% Top-3 (28.22% Macro-F1)** trên **236 ảnh thực tế ngoài trời (`PlantDoc Test`)**.
- **Giải pháp:**
  1. Khởi tạo từ checkpoint tốt nhất `models/convnext_tiny_khoi.pth`.
  2. Trộn `PlantVillage/train` (70.295 ảnh) + `PlantDoc/train` (2.336 ảnh ngoài trời, **oversample ~10x** để chiếm ~25% mỗi batch).
  3. Bật **Field-Robust Augmentation** (`RandomPerspective`, `ColorJitter` mạnh hơn, `RandomErasing` mô phỏng che khuất lá/cành).
  4. Chấm song song trên **cả 2 tập**: `PlantVillage Valid` (Lab) và `PlantDoc Test` (Ngoài trời) sau mỗi epoch, xuất ra `models/convnext_tiny_field_v2_int8.onnx`.

In [ ]:
import os, sys, json, copy, time, importlib, subprocess
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ConcatDataset, DataLoader
from torchvision import datasets, models, transforms

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ) or Path("/kaggle/working").exists()
GITHUB_REPO = "https://github.com/Longcodepro/plant-disease-ml.git"
GITHUB_BRANCH = "feat/convnext-onnx-rust-khoi"

def sync_github_repo(target_dir: Path) -> None:
    if not (target_dir / ".git").exists():
        clone = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, str(target_dir)],
            capture_output=True, text=True,
        )
        if clone.returncode != 0:
            subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(target_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(target_dir), "pull", "--ff-only"], check=False)

if IN_COLAB or IN_KAGGLE:
    import importlib.util
    if importlib.util.find_spec("onnx") is None or importlib.util.find_spec("onnxruntime") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = Path("/content/plant-disease-ml")
    sync_github_repo(PROJECT_DIR)
    os.chdir(PROJECT_DIR)
    importlib.invalidate_caches()

    preferred_drive = Path("/content/drive/MyDrive/Mac_PhamGiaKhoi/Dự án/ML/plant-disease-ml")
    DRIVE_OUT_DIR = preferred_drive if preferred_drive.exists() else Path("/content/drive/MyDrive/plant-disease-ml-outputs")
elif IN_KAGGLE:
    PROJECT_DIR = Path("/tmp/plant-disease-ml")
    sync_github_repo(PROJECT_DIR)
    os.chdir(PROJECT_DIR)
    importlib.invalidate_caches()
    DRIVE_OUT_DIR = Path("/kaggle/working")
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(PROJECT_DIR)
    DRIVE_OUT_DIR = PROJECT_DIR

OUT_MODELS_DIR = DRIVE_OUT_DIR / "models"
OUT_REPORTS_DIR = (DRIVE_OUT_DIR / "eda_reports") if IN_KAGGLE else (DRIVE_OUT_DIR / "notebooks" / "eda_reports")
OUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("IN_COLAB       :", IN_COLAB, "| IN_KAGGLE:", IN_KAGGLE)
print("PROJECT_DIR    :", PROJECT_DIR)
print("OUT_MODELS_DIR :", OUT_MODELS_DIR)
print("OUT_REPORTS_DIR:", OUT_REPORTS_DIR)
print("DEVICE         :", device, (f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

In [ ]:
# 1. Chuẩn bị cả PlantVillage (Lab) và PlantDoc (In-the-Wild) trên SSD tạm (/content/data hoặc /tmp/data)
if IN_COLAB:
    DATA_ROOT = Path("/content/data")
elif IN_KAGGLE:
    DATA_ROOT = Path("/tmp/data")
else:
    DATA_ROOT = PROJECT_DIR / "data"

if (IN_COLAB or IN_KAGGLE) and not (DATA_ROOT / "train").exists():
    from scripts.download_data import DEFAULT_GDRIVE_FILE_ID, download_from_gdrive, extract_and_organize
    drive_archive = DRIVE_OUT_DIR / "data.tar.gz"
    if drive_archive.exists():
        extract_and_organize(drive_archive, DATA_ROOT)
    else:
        tmp_arc = Path("/tmp/data.tar.gz")
        download_from_gdrive(DEFAULT_GDRIVE_FILE_ID, tmp_arc)
        extract_and_organize(tmp_arc, DATA_ROOT)
        tmp_arc.unlink(missing_ok=True)

PLANTDOC_DIR = DATA_ROOT / "plantdoc"
if not (PLANTDOC_DIR / "train").exists():
    from scripts.prepare_plantdoc import prepare_plantdoc
    prepare_plantdoc(DATA_ROOT / "plantdoc_raw", PLANTDOC_DIR, PROJECT_DIR / "models" / "class_indices.json")

# Tạo bản chia sạch 0% UUID leakage ngay trên SSD tạm (~2 giây bằng symlink, không làm phình Output Kaggle)
CLEAN_PV_DIR = DATA_ROOT / "clean_split"
if not (CLEAN_PV_DIR / "train").exists():
    from scripts.audit_uuid_leakage import audit_and_optionally_split
    audit_and_optionally_split(
        data_dir=DATA_ROOT,
        out_json=OUT_REPORTS_DIR / "uuid_leakage_audit.json",
        build_clean_split=True,
        clean_dir=CLEAN_PV_DIR,
    )

print("PlantVillage Clean (0% leakage):", CLEAN_PV_DIR)
print("PlantDoc train    :", PLANTDOC_DIR / "train")

In [ ]:
# 2. Tạo Mixed-Domain DataLoader + Augmentation mô phỏng thực tế ngoài đồng ruộng
from notebooks.data_pipeline import IMAGENET_MEAN, IMAGENET_STD, get_eval_transforms
from notebooks.eval_engine import evaluate_model, print_evaluation_summary

IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 6
LR = 5e-5
PLANTDOC_OVERSAMPLE = 10  # 2.336 * 10 = 23.360 ảnh (~25% mỗi batch là ảnh ngoài trời)
NUM_WORKERS = 4 if (IN_KAGGLE and (os.cpu_count() or 2) >= 4) else 2

field_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.65, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=25),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

pv_train_ds = datasets.ImageFolder(str(CLEAN_PV_DIR / "train"), transform=field_train_transform)
pd_train_ds = datasets.ImageFolder(str(PLANTDOC_DIR / "train"), transform=field_train_transform)
# Đồng nhất class_to_idx của PlantDoc (28 lớp) về đúng 38 lớp của PlantVillage
pd_train_ds.samples = [(path, pv_train_ds.class_to_idx[pd_train_ds.classes[cls_idx]]) for path, cls_idx in pd_train_ds.samples]
pd_train_ds.targets = [s[1] for s in pd_train_ds.samples]

mixed_train_ds = ConcatDataset([pv_train_ds] + [pd_train_ds] * PLANTDOC_OVERSAMPLE)
train_loader = DataLoader(
    mixed_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    drop_last=True,
)

pv_valid_ds = datasets.ImageFolder(str(CLEAN_PV_DIR / "valid"), transform=get_eval_transforms(IMAGE_SIZE))
pv_valid_loader = DataLoader(pv_valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

pd_test_ds = datasets.ImageFolder(str(PLANTDOC_DIR / "test"), transform=get_eval_transforms(IMAGE_SIZE))
pd_test_ds.samples = [(path, pv_train_ds.class_to_idx[pd_test_ds.classes[cls_idx]]) for path, cls_idx in pd_test_ds.samples]
pd_test_ds.targets = [s[1] for s in pd_test_ds.samples]
pd_test_loader = DataLoader(pd_test_ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

print(f"Mixed Train Samples: {len(mixed_train_ds):,} (PV {len(pv_train_ds):,} + PlantDoc {len(pd_train_ds):,} x {PLANTDOC_OVERSAMPLE})")
print(f"Lab Valid: {len(pv_valid_ds):,} | Field Test (PlantDoc): {len(pd_test_ds):,}")

In [ ]:
# 3. Load Checkpoint Lab (nếu có) hoặc ImageNet pretrained với Differential LR & Fine-tune Domain Adaptation
BASE_CKPT_CANDIDATES = [
    OUT_MODELS_DIR / "convnext_tiny_khoi.pth",
    PROJECT_DIR / "models" / "convnext_tiny_khoi.pth",
]
BASE_CKPT = next((p for p in BASE_CKPT_CANDIDATES if p.exists()), None)

SAVE_V2_PTH = OUT_MODELS_DIR / "convnext_tiny_field_v2.pth"
SAVE_V2_ONNX = OUT_MODELS_DIR / "convnext_tiny_field_v2.onnx"
SAVE_V2_INT8 = OUT_MODELS_DIR / "convnext_tiny_field_v2_int8.onnx"

num_classes = len(pv_train_ds.classes)
if BASE_CKPT is not None:
    print(f"📥 Nạp Lab Checkpoint: {BASE_CKPT}")
    model = models.convnext_tiny(weights=None)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    ckpt = torch.load(BASE_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device)
    RUN_EPOCHS = EPOCHS
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
else:
    print("⚠️ Chưa có convnext_tiny_khoi.pth -> khởi tạo ConvNeXt_Tiny_Weights.DEFAULT (ImageNet) + Differential LR (8 epochs)")
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    model = model.to(device)
    RUN_EPOCHS = 8
    head_params = list(model.classifier.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = optim.AdamW(
        [
            {"params": backbone_params, "lr": 1e-4},
            {"params": head_params, "lr": 5e-4},
        ],
        weight_decay=1e-2,
    )

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=RUN_EPOCHS, eta_min=1e-6)
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_score = 0.0
for ep in range(1, RUN_EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in train_loader:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)
    scheduler.step()

    lab_m = evaluate_model(model, pv_valid_loader, device=device)
    field_m = evaluate_model(model, pd_test_loader, device=device)
    # Chọn checkpoint tối ưu đồng thời cả Lab & Ngoài trời
    combined = 0.4 * lab_m["macro_f1"] + 0.6 * field_m["macro_f1"]
    print(
        f"[DA {ep}/{RUN_EPOCHS}] ({time.time()-t0:.1f}s) | Train Acc {correct/total*100:.2f}% | "
        f"Lab Acc {lab_m['accuracy']*100:.2f}% | Field (PlantDoc) Acc {field_m['accuracy']*100:.2f}% Macro-F1 {field_m['macro_f1']*100:.2f}%"
    )
    if combined > best_score:
        best_score = combined
        torch.save({
            "model_name": "convnext_tiny",
            "author": "khoi_domain_adapted",
            "num_classes": num_classes,
            "val_accuracy": lab_m["accuracy"],
            "val_f1_macro": lab_m["macro_f1"],
            "field_accuracy": field_m["accuracy"],
            "field_f1_macro": field_m["macro_f1"],
            "model_state_dict": model.state_dict(),
        }, SAVE_V2_PTH)
        print(f"   ⭐ Lưu Best Domain-Adapted Checkpoint -> {SAVE_V2_PTH}")

In [ ]:
# 3B. Stage 2 Chuyên sâu PlantDoc (80% PlantDoc Balanced + CutMix/MixUp + EMA + Cooldown)
import shutil
import numpy as np
from torch.utils.data import Subset, WeightedRandomSampler

with open(PROJECT_DIR / "models" / "class_indices.json", encoding="utf-8") as f:
    class_to_idx = json.load(f)["class_to_idx"]
num_classes = len(class_to_idx)

STAGE1_BACKUP = OUT_MODELS_DIR / "convnext_tiny_field_v2_stage1.pth"
if SAVE_V2_PTH.exists() and not STAGE1_BACKUP.exists():
    shutil.copy2(SAVE_V2_PTH, STAGE1_BACKUP)

ckpt_src = STAGE1_BACKUP if STAGE1_BACKUP.exists() else SAVE_V2_PTH
ckpt_s1 = torch.load(ckpt_src, map_location=device, weights_only=False)
stage1_field_f1 = float(ckpt_s1.get("field_f1_macro", 0.0))
stage1_field_acc = float(ckpt_s1.get("field_accuracy", 0.0))

pdoc_strong_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.50, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomApply([transforms.RandomAffine(degrees=30, translate=(0.08, 0.08), shear=10)], p=0.65),
    transforms.RandomPerspective(distortion_scale=0.22, p=0.35),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.07),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.4))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.30, scale=(0.02, 0.18)),
])

pdoc_clean_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.68, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.04),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

pd_train_p3 = datasets.ImageFolder(str(PLANTDOC_DIR / "train"), transform=pdoc_strong_tf)
pd_train_p3.samples = [(p, class_to_idx[pd_train_p3.classes[y]]) for p, y in pd_train_p3.samples]
pd_train_p3.targets = [y for _, y in pd_train_p3.samples]

# Replay 200 ảnh/lớp x 38 lớp từ Clean PlantVillage để giữ nguyên 10 lớp không có trong PlantDoc
import random as _py_random
pv_replay_full = datasets.ImageFolder(str(CLEAN_PV_DIR / "train"), transform=pdoc_strong_tf)
rng_p3 = _py_random.Random(42)
by_cls = {}
for idx_s, (_, y) in enumerate(pv_replay_full.samples):
    by_cls.setdefault(y, []).append(idx_s)
replay_indices = []
for y, idxs in sorted(by_cls.items()):
    replay_indices.extend(rng_p3.sample(idxs, min(200, len(idxs))))
pv_replay_sub = Subset(pv_replay_full, replay_indices)

pd_cls_counts = {}
for y in pd_train_p3.targets:
    pd_cls_counts[y] = pd_cls_counts.get(y, 0) + 1
pv_cls_counts = {}
for idx_s in replay_indices:
    y = pv_replay_full.targets[idx_s]
    pv_cls_counts[y] = pv_cls_counts.get(y, 0) + 1

n_pd_cls, n_pv_cls = len(pd_cls_counts), len(pv_cls_counts)
sample_weights = (
    [0.80 / (n_pd_cls * pd_cls_counts[y]) for y in pd_train_p3.targets]
    + [0.20 / (n_pv_cls * pv_cls_counts[pv_replay_full.targets[idx_s]]) for idx_s in replay_indices]
)

p3_dataset = ConcatDataset([pd_train_p3, pv_replay_sub])
SAMPLES_PER_EPOCH = 22400
sampler = WeightedRandomSampler(sample_weights, num_samples=SAMPLES_PER_EPOCH, replacement=True)
p3_loader = DataLoader(
    p3_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=False,
    drop_last=True,
)

model = models.convnext_tiny(weights=None)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
model.load_state_dict(ckpt_s1["model_state_dict"])
model = model.to(device)

eval_net = copy.deepcopy(model).to(device)
ema_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
EMA_DECAY = 0.995

P3_EPOCHS = 8  # 5 epoch CutMix/MixUp + 3 epoch Cooldown sạch (~42-45 phút trên T4)
head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": 4e-5},
        {"params": head_params, "lr": 2e-4},
    ],
    weight_decay=2e-2,
)
warmup_epochs = 1
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        optim.lr_scheduler.LinearLR(optimizer, start_factor=0.2, total_iters=warmup_epochs),
        optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=P3_EPOCHS - warmup_epochs, eta_min=1e-6),
    ],
    milestones=[warmup_epochs],
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def apply_mix_batch(imgs, targs, crit, out_fn):
    p = np.random.rand()
    if p < 0.35:
        lam = float(np.random.beta(1.0, 1.0))
        perm = torch.randperm(imgs.size(0), device=imgs.device)
        _, _, H, W = imgs.shape
        cut_rat = float(np.sqrt(max(0.0, 1.0 - lam)))
        cw, ch = int(W * cut_rat), int(H * cut_rat)
        cx, cy = np.random.randint(W), np.random.randint(H)
        x1, x2 = int(np.clip(cx - cw // 2, 0, W)), int(np.clip(cx + cw // 2, 0, W))
        y1, y2 = int(np.clip(cy - ch // 2, 0, H)), int(np.clip(cy + ch // 2, 0, H))
        if x2 > x1 and y2 > y1:
            patch = imgs[perm, :, y1:y2, x1:x2].clone()
            imgs[:, :, y1:y2, x1:x2] = patch
            lam = 1.0 - ((x2 - x1) * (y2 - y1) / float(H * W))
        outs = out_fn(imgs)
        return outs, lam * crit(outs, targs) + (1.0 - lam) * crit(outs, targs[perm])
    elif p < 0.60:
        lam = float(np.random.beta(0.25, 0.25))
        perm = torch.randperm(imgs.size(0), device=imgs.device)
        imgs = lam * imgs + (1.0 - lam) * imgs[perm]
        outs = out_fn(imgs)
        return outs, lam * crit(outs, targs) + (1.0 - lam) * crit(outs, targs[perm])
    outs = out_fn(imgs)
    return outs, crit(outs, targs)

SAVE_V3_PTH = OUT_MODELS_DIR / "convnext_tiny_field_v3.pth"
best_p3_score = 0.0
best_v3_metrics = None

print(f"🚀 Bắt đầu Stage 2 chuyên sâu PlantDoc từ Stage 1 (PlantDoc Acc {stage1_field_acc*100:.2f}% | Macro-F1 {stage1_field_f1*100:.2f}%)")
print(f"   Samples/epoch: {SAMPLES_PER_EPOCH:,} (80% PlantDoc cân bằng 28 lớp + 20% PlantVillage 38 lớp) | Epochs: {P3_EPOCHS}")

for ep in range(1, P3_EPOCHS + 1):
    t0 = time.time()
    use_mix = (ep <= P3_EPOCHS - 3)
    if not use_mix and pd_train_p3.transform is not pdoc_clean_tf:
        pd_train_p3.transform = pdoc_clean_tf
        pv_replay_full.transform = pdoc_clean_tf

    model.train()
    for images, targets in p3_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            if use_mix:
                outputs, loss = apply_mix_batch(images, targets, criterion, model)
            else:
                outputs = model(images)
                loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.is_floating_point():
                    ema_state[k].mul_(EMA_DECAY).add_(v.detach(), alpha=1.0 - EMA_DECAY)
                else:
                    ema_state[k].copy_(v)
    scheduler.step()

    eval_net.load_state_dict(ema_state)
    lab_m = evaluate_model(eval_net, pv_valid_loader, device=device)
    field_m = evaluate_model(eval_net, pd_test_loader, device=device)
    score = (0.20 * lab_m["macro_f1"] + 0.80 * field_m["macro_f1"]) if lab_m["accuracy"] >= 0.95 else 0.0
    print(
        f"[PlantDoc-S2 {ep:02d}/{P3_EPOCHS}] ({time.time()-t0:.1f}s) | Mix={'ON' if use_mix else 'OFF'} | "
        f"Lab Acc {lab_m['accuracy']*100:.2f}% | Field Acc {field_m['accuracy']*100:.2f}% Macro-F1 {field_m['macro_f1']*100:.2f}%"
    )
    if score > best_p3_score:
        best_p3_score = score
        best_v3_metrics = (lab_m, field_m)
        torch.save({
            "model_name": "convnext_tiny",
            "author": "khoi_domain_adapted_v3",
            "num_classes": num_classes,
            "val_accuracy": lab_m["accuracy"],
            "val_f1_macro": lab_m["macro_f1"],
            "field_accuracy": field_m["accuracy"],
            "field_f1_macro": field_m["macro_f1"],
            "model_state_dict": eval_net.state_dict(),
        }, SAVE_V3_PTH)
        print(f"   ⭐ Lưu Best PlantDoc-S2 EMA Checkpoint -> {SAVE_V3_PTH.name}")

if best_v3_metrics is not None:
    ckpt_v3 = torch.load(SAVE_V3_PTH, map_location=device, weights_only=False)
    eval_net.load_state_dict(ckpt_v3["model_state_dict"])
    eval_net.eval()

    tta_correct, tta_total = 0, 0
    with torch.no_grad():
        for imgs, targs in pd_test_loader:
            imgs = imgs.to(device)
            logits = 0.5 * (eval_net(imgs) + eval_net(torch.flip(imgs, dims=[3])))
            tta_correct += (logits.argmax(dim=1).cpu() == targs).sum().item()
            tta_total += targs.size(0)
    print(f"🔎 PlantDoc Test với Flip-TTA (trên {SAVE_V3_PTH.name}): {tta_correct}/{tta_total} ({tta_correct/tta_total*100:.2f}%)")

    if ckpt_v3["field_f1_macro"] >= stage1_field_f1:
        shutil.copy2(SAVE_V3_PTH, SAVE_V2_PTH)
        print(f"✅ Đã vượt Stage 1 ({stage1_field_f1*100:.2f}% -> {ckpt_v3['field_f1_macro']*100:.2f}%)! Cập nhật {SAVE_V2_PTH.name} để Cell #4 xuất ONNX luôn.")
    else:
        print(f"ℹ️ Stage 1 vẫn cao hơn ({stage1_field_f1*100:.2f}% vs {ckpt_v3['field_f1_macro']*100:.2f}%), giữ nguyên {SAVE_V2_PTH.name} cho Cell #4.")

In [ ]:
# 4. Xuất ONNX FP32 + INT8 & chạy script đánh giá Domain Shift chính thức
ckpt_v2 = torch.load(SAVE_V2_PTH, map_location="cpu", weights_only=False)
model_cpu = models.convnext_tiny(weights=None)
model_cpu.classifier[2] = nn.Linear(model_cpu.classifier[2].in_features, num_classes)
model_cpu.load_state_dict(ckpt_v2["model_state_dict"])
model_cpu.eval()

dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
torch.onnx.export(
    model_cpu, dummy, str(SAVE_V2_ONNX),
    export_params=True, opset_version=17, do_constant_folding=True,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
    dynamo=False,
)

from scripts.quantize_onnx import quantize_model
quantize_model(SAVE_V2_ONNX, SAVE_V2_INT8, mode="dynamic")
print(f"📦 ONNX FP32: {SAVE_V2_ONNX.stat().st_size/1024**2:.1f} MB -> INT8: {SAVE_V2_INT8.stat().st_size/1024**2:.1f} MB")

from scripts.eval_domain_shift import evaluate_on_plantdoc
evaluate_on_plantdoc(
    SAVE_V2_INT8,
    PLANTDOC_DIR / "test",
    PROJECT_DIR / "models" / "class_indices.json",
    OUT_REPORTS_DIR / "plantdoc_domain_shift_v2.json",
)